# Optimal transport of the case effects to Thailand

**Purpose.** This notebook moves the estimated effects of the theme-park and resort openings from the source episodes to the feature cloud of the target economy with unbalanced optimal transport whose ground cost is weighted by the learned feature importance. Episodes that resemble the target on the features that modify the effect receive the most mass. The notebook describes the target economy once, for this notebook and for the next one, and validates the transport rule by leaving out one economy at a time, relearning the feature importance from the remaining economies only, and predicting the episodes of the left-out economy. It then computes the transport plan for the target, checks that the target lies inside the support of the sources, tests the overlap between the target and the sources, and computes the transported effect, its sampling distribution, a conformal predictive interval built from the validation errors, and the transported placebo noise that serves as the null distribution of the detection analysis. The transported effects are saved and are not printed in this notebook: the next notebook states the route decision first and shows every effect for Thailand after it.

**Input files.** data/processed/global_panel.csv, the Thailand baseline and proposal files that the run context supplies, and in the results directory case_effects.csv, case_placebos.csv, case_features_imputed.csv, reference_features.csv and transport_weights.csv.

**Output files.** In the results directory: loco_table.csv, loco_summary.csv, loco_ratios.csv, loco_table_rel.csv, loco_summary_rel.csv, loco_ratios_rel.csv, loco_fold_weights.csv, transport_plan.csv, transport_usage.csv, transport_sensitivity.csv, thailand_target_baseline.csv, thailand_transport.json, thailand_transport_draws.csv, thailand_scenario_draws.csv, thailand_null_draws.csv and overlap_test.json. In the figures directory: ot_loco_validation.png, ot_plan_heatmap.png, ot_feature_space.png and ot_target_support.png.

**How to run.** From the repository root, run `python -m nbconvert --to notebook --execute --ExecutePreprocessor.kernel_name=dtt-venv --output-dir executed notebooks/04_optimal_transport.ipynb`, which writes the executed copy to the folder `executed` and leaves the notebook itself unchanged. The optional environment variables `DTT_RESULTS_DIR` and `DTT_FIGURE_DIR` name other directories for the tables and the figures, and `DTT_GUARD=0` switches the thermal guard off.

**Run time.** About 40 seconds on a PC with 12 threads, most of it in the relearning of the importance weights in the validation folds and in the bootstrap of the transported effect.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_root = next(p for p in (_here, *_here.parents) if (p / "src" / "dtt" / "__init__.py").is_file())
sys.path.insert(0, str(_root / "src"))

from dtt import workflow

ctx = workflow.setup_run("04_optimal_transport")
ctx.describe()

In [ ]:
import hashlib
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.text import Text
from matplotlib.transforms import Bbox

from dtt import cases, impact, importance, transport
from dtt import panel as panel_module

NOTEBOOK_START = time.perf_counter()
INPUT_TABLES = (
    "case_effects",
    "case_placebos",
    "case_features_imputed",
    "reference_features",
    "transport_weights",
)
PANEL_PATH = ctx.processed_dir / "global_panel.csv"
if not PANEL_PATH.is_file():
    raise FileNotFoundError(f"{PANEL_PATH} not found; run 00_build_global_panel first")
ctx.require(*INPUT_TABLES, PANEL_PATH, ctx.baseline_path, ctx.proposal_path)
INPUT_DIGESTS = {name: hashlib.sha256(ctx.table_path(name).read_bytes()).hexdigest() for name in INPUT_TABLES}
print("All input files found.")

## Settings

The settings cell fixes every setting once: repeat counts, bootstrap sizes and the number of permutations of the importance estimate, together with the other constants of the method.

In [ ]:
# Settings
N_BOOT_LOCO = 5000  # bootstrap resamples of the descriptive intervals of the validation RMSE ratios
N_BOOT_TRANSPORT = 5000  # cluster bootstrap replicates of the transported effect, and predictive draws
N_PERM = 9999  # permutations of the overlap test
N_NULL = 20000  # draws from the transported placebo distribution
IMPORTANCE_REPEATS = 5  # permutations per feature and fold when the importance is relearned in a fold
MIN_FOLD_ECONOMIES = 4  # fewest training economies for which the importance is fitted in a fold
RHO_SOURCE = 1.0  # strength of the penalty on the source marginal
RHO_SENSITIVITY = (0.3, 3.0)  # other strengths of that penalty
CLIP = 5.0  # bound of the standardised feature values
COST_YEAR_ALTERNATIVE = 2019  # year of the GDP and the exchange rate in the alternative reading of the cost share
RATIO_YEARS = tuple(range(2015, 2020))  # years whose ratio of panel to baseline receipts puts the other baseline years on the panel basis
TARGET_YEARS = tuple(range(2015, 2020))  # years of the yearly feature vectors of the target economy
RECEIPTS_WINDOW = 3  # last years of the target years whose mean receipts share of GDP describes the target
PRE_YEARS = 3  # pre-opening years in the cloud of each validation episode
MIN_REFERENCE_ROWS = 20  # rows with a value that a reference feature needs to define a scale
PREDICTIVE_METHOD = "ot_weighted"  # validated predictor whose errors give the predictive distribution
CONFORMAL_LEVEL = 0.90  # nominal coverage of the conformal interval
MIN_ESS = 2.0  # fewest effective sources behind the target for the target to count as supported
MIN_WEIGHT = 0.10  # smallest weight of a feature whose range test must pass in the support check
RANGE_TOLERANCE = 0.10  # margin around the range of the sources, as a share of that range
PROBS = (0.05, 0.25, 0.5, 0.75, 0.95)  # reported quantile levels

SEED_LOCO = ctx.seeds["transport"]
SEED_BOOT = ctx.seeds["transport bootstrap"]
SEED_PERM = ctx.seeds["overlap test"]
SEED_NULL = ctx.seeds["transport"]
SEED_IMPORTANCE = ctx.seeds["importance"]

print(f"Bootstrap resamples: validation {N_BOOT_LOCO}, transported effect and predictive draws {N_BOOT_TRANSPORT}; permutations {N_PERM}; null draws {N_NULL}")
print(f"Importance relearned in every validation fold: {IMPORTANCE_REPEATS} permutations per feature and fold, light configuration, at least {MIN_FOLD_ECONOMIES} training economies")
print(f"Source penalty rho: {RHO_SOURCE} with sensitivity to {RHO_SENSITIVITY}; target marginal enforced")
print(f"Standardised feature values are clipped at plus and minus {CLIP:g}")
print(f"Target years: {TARGET_YEARS[0]} to {TARGET_YEARS[-1]}; receipts share of the target: mean of the last {RECEIPTS_WINDOW} of them; alternative cost year {COST_YEAR_ALTERNATIVE}")
print(f"Years of the ratio of panel to baseline receipts: {RATIO_YEARS[0]} to {RATIO_YEARS[-1]}")
print(f"Predictive interval: conformal interval at level {CONFORMAL_LEVEL:g} from the errors of {PREDICTIVE_METHOD} in the leave-one-economy-out validation")
print(f"Support check: at least {MIN_ESS:g} effective sources; range test on every feature with weight of at least {MIN_WEIGHT:g}; margin of {RANGE_TOLERANCE:g} of the range of the sources")
print(f"Seeds: validation {SEED_LOCO}, bootstrap {SEED_BOOT}, overlap test {SEED_PERM}, null draws {SEED_NULL}, importance {SEED_IMPORTANCE}")
print(f"A reference feature needs at least {MIN_REFERENCE_ROWS} rows with a value to define its own centre and scale")

## Helper functions

In [ ]:
def fill_cloud(cloud: pd.DataFrame, features: list[str], fallback: pd.Series) -> tuple[pd.DataFrame, int]:
    """Fill the missing entries of a cloud of yearly feature vectors.

    A missing entry takes the median of the finite values of the same feature in the other rows of the cloud, and
    the value in ``fallback`` when no row has a finite value.

    Parameters
    ----------
    cloud : pandas.DataFrame
        One row per year with the columns in ``features``.
    features : list of str
        Features to keep, in order.
    fallback : pandas.Series
        Value for a feature that is missing in every row, indexed by feature name.

    Returns
    -------
    filled : pandas.DataFrame
        The cloud restricted to ``features`` without missing values, with the index of ``cloud``.
    n_filled : int
        Number of entries that were filled.
    """
    values = cloud[features].astype(float).copy()
    n_filled = int(values.isna().to_numpy().sum())
    for name in features:
        column = values[name]
        if column.isna().any():
            finite = column.dropna()
            values[name] = column.fillna(float(finite.median()) if len(finite) else float(fallback[name]))
    return values, n_filled


def standardise(table: pd.DataFrame, columns: list[str] | None = None, clip: float | None = CLIP) -> pd.DataFrame:
    """Put a feature table on the scale of the reference distribution.

    Every table is standardised against the same reference and the same pooled episodes, so that sources, validation
    clouds and target clouds share one centre and one scale for each feature.

    Parameters
    ----------
    table : pandas.DataFrame
        Raw features, one row per source or target point.
    columns : list of str, optional
        Features to return, in order; all features of the analysis by default.
    clip : float or None
        Bound of the standardised values; ``None`` leaves them unclipped.

    Returns
    -------
    pandas.DataFrame
        Features centred at the reference median and divided by the scaled median absolute deviation of the
        reference, with the index of ``table``. A feature that the reference cannot describe takes its centre and
        scale from the pooled episodes.
    """
    cols = FEATURES if columns is None else columns
    _, z, _ = transport.robust_standardise(POOLED, table, reference=REFERENCE, columns=cols, clip=clip)
    return z


def count_clipped(table: pd.DataFrame, columns: list[str] | None = None) -> tuple[int, int]:
    """Count the standardised values that the clip bound changes.

    Parameters
    ----------
    table : pandas.DataFrame
        Raw features, one row per source or target point.
    columns : list of str, optional
        Features to count; all features of the analysis by default.

    Returns
    -------
    n_clipped : int
        Number of finite standardised values whose absolute value exceeds the clip bound.
    n_values : int
        Number of finite standardised values.
    """
    values = standardise(table, columns, clip=None).to_numpy(dtype=float)
    finite = np.isfinite(values)
    return int((np.abs(values[finite]) > CLIP).sum()), int(finite.sum())


class FoldImportance:
    """Feature weights relearned inside each fold of the leave-one-economy-out validation.

    An object of this class is the weight function of ``transport.loco_validation``. For the fold of one held-out
    economy it fits the penalty-averaged importance of the effect on the training episodes alone, so that no
    information about the held-out effects reaches the weights of the fold. A fold in which the importance cannot be
    fitted receives uniform weights and is recorded. The weights of a fold are computed once and kept.

    Parameters
    ----------
    features : pandas.DataFrame
        Raw features of the source episodes, one row per episode, in the order of the standardised sources.
    effect : numpy.ndarray
        Effect of each episode in percentage points of GDP.
    groups : numpy.ndarray
        Economy of each episode.
    n_repeats : int
        Permutations per feature and fold.
    seed : int
        Seed of the importance estimate.
    min_economies : int
        Fewest training economies for which the importance is fitted.
    guard : callable or None
        Thermal guard passed to the importance estimate.
    """

    def __init__(
        self,
        features: pd.DataFrame,
        effect: np.ndarray,
        groups: np.ndarray,
        n_repeats: int,
        seed: int,
        min_economies: int,
        guard: object,
    ) -> None:
        self.features = features
        self.effect = np.asarray(effect, dtype=float)
        self.groups = np.asarray(groups)
        self.n_repeats = int(n_repeats)
        self.seed = int(seed)
        self.min_economies = int(min_economies)
        self.guard = guard
        self.names = list(features.columns)
        self.records: dict[str, dict[str, object]] = {}

    def __call__(self, train_idx: np.ndarray) -> pd.Series:
        """Weights of the fold whose training episodes sit at the positions ``train_idx``.

        Parameters
        ----------
        train_idx : numpy.ndarray
            Integer positions of the episodes outside the held-out economy.

        Returns
        -------
        pandas.Series
            Non-negative weights summing to one, indexed by feature.
        """
        train = np.asarray(train_idx, dtype=int)
        held_out = sorted(set(self.groups.tolist()) - set(self.groups[train].tolist()))
        label = "+".join(str(g) for g in held_out)
        if label in self.records:
            return self.records[label]["weights"]
        n_economies = len(set(self.groups[train].tolist()))
        weights = pd.Series(1.0 / len(self.names), index=self.names)
        fitted, reason = False, ""
        if n_economies < self.min_economies:
            reason = f"{n_economies} training economies"
        else:
            try:
                result = importance.lambda_averaged_importance(
                    self.features.iloc[train],
                    self.effect[train],
                    self.groups[train],
                    light=True,
                    compute_diagnostics=False,
                    cv_repeats=1,
                    n_repeats=self.n_repeats,
                    random_state=self.seed,
                    guard=self.guard,
                )
                weights = pd.Series(result.weights(0.0), index=result.feature_names)
                fitted = True
            except ValueError as error:
                reason = str(error)
        self.records[label] = {"weights": weights, "fitted": fitted, "reason": reason}
        return weights

    def table(self) -> pd.DataFrame:
        """Weights of every fold in long form.

        Returns
        -------
        pandas.DataFrame
            One row per fold and feature with the columns ``held_out_group``, ``feature``, ``weight`` and ``fitted``
            (false for a fold with uniform fallback weights).
        """
        rows = [
            {"held_out_group": label, "feature": feature, "weight": float(weight), "fitted": bool(record["fitted"])}
            for label, record in self.records.items()
            for feature, weight in record["weights"].items()
        ]
        return pd.DataFrame(rows, columns=["held_out_group", "feature", "weight", "fitted"])


def solve_transport(Zs: pd.DataFrame, Zt: pd.DataFrame, weights: pd.Series, rho: float | None) -> dict[str, object]:
    """Solve the unbalanced transport problem from the sources to a target cloud.

    The sources and the target points carry uniform mass; the source marginal is relaxed with strength ``rho`` and
    the target marginal is enforced. The regularisation strength is ``select_eps`` of the sources.

    Parameters
    ----------
    Zs : pandas.DataFrame
        Standardised sources, indexed by case.
    Zt : pandas.DataFrame
        Standardised target points, indexed by year.
    weights : pandas.Series
        Feature weights, indexed by feature name.
    rho : float or None
        Strength of the penalty on the source marginal.

    Returns
    -------
    dict
        ``plan`` (the transport plan), ``eps`` (regularisation strength), ``b`` (target masses) and ``cost``
        (ground cost matrix as a DataFrame).
    """
    eps = transport.select_eps(Zs, weights)
    cost = pd.DataFrame(transport.weighted_sq_cost(Zs, Zt, weights), index=Zs.index, columns=Zt.index)
    n, m = cost.shape
    a = pd.Series(np.full(n, 1.0 / n), index=cost.index)
    b = pd.Series(np.full(m, 1.0 / m), index=cost.columns)
    plan = transport.sinkhorn_plan(a, b, cost, eps, rho_source=rho, rho_target=None, max_iter=20000)
    return {"plan": plan, "eps": eps, "b": b.to_numpy(), "cost": cost}


def principal_axes(matrix: np.ndarray, n_components: int = 2) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Principal axes of a centred matrix by the singular value decomposition.

    Parameters
    ----------
    matrix : numpy.ndarray
        Rows are points and columns are features.
    n_components : int
        Number of axes to return.

    Returns
    -------
    centre : numpy.ndarray
        Column means of ``matrix``.
    axes : numpy.ndarray
        Loadings, one row per component; the sign of each axis makes its largest absolute loading positive.
    share : numpy.ndarray
        Share of the total variance explained by each returned component.
    """
    centre = matrix.mean(axis=0)
    _, singular, vt = np.linalg.svd(matrix - centre, full_matrices=False)
    axes = vt[:n_components].copy()
    for k in range(axes.shape[0]):
        if axes[k, np.argmax(np.abs(axes[k]))] < 0:
            axes[k] = -axes[k]
    variance = singular**2
    return centre, axes, (variance / variance.sum())[:n_components]


def quantile_dict(values: np.ndarray, probs: tuple[float, ...] = PROBS) -> dict[str, float]:
    """Quantiles of a sample as a dictionary with the keys ``p5``, ``p25`` and so on."""
    return {f"p{round(100 * p):g}": float(q) for p, q in zip(probs, np.quantile(values, probs))}


def show(table: pd.DataFrame, digits: int = 4) -> None:
    """Print a table with a fixed number of decimals."""
    print(table.round(digits).to_string())

In [ ]:
def interval_record(interval: dict[str, object]) -> dict[str, object]:
    """Plain dictionary of a conformal interval.

    Parameters
    ----------
    interval : dict
        Output of ``transport.loco_interval`` or the entry ``interval`` of ``transport.loco_predictive_draws``.

    Returns
    -------
    dict
        The entries ``lower``, ``upper``, ``half_width``, ``n``, ``rank``, ``level``, ``guaranteed_level`` and
        ``status``; the bounds are missing when the interval does not exist.
    """
    keys = ("lower", "upper", "half_width", "n", "rank", "level", "guaranteed_level", "status")
    return {key: interval[key] for key in keys}


def support_record(result: dict[str, object]) -> dict[str, object]:
    """Plain dictionary of the result of ``transport.target_support``.

    Parameters
    ----------
    result : dict
        Output of ``transport.target_support``.

    Returns
    -------
    dict
        The scalar entries of the result, the settings of the check, the reasons and the table of features as a
        dictionary keyed by feature.
    """
    scalar_keys = ("ess", "max_source_share", "n_sources", "weighted_outside_share", "ess_ok", "range_ok", "supported")
    record = {key: result[key] for key in scalar_keys}
    record["reasons"] = list(result["reasons"])
    record["settings"] = {"min_ess": MIN_ESS, "min_weight": MIN_WEIGHT, "range_tolerance": RANGE_TOLERANCE, "clip": CLIP}
    record["features"] = result["features"].to_dict(orient="index")
    return record


def cost_share(scenario_row: dict[str, object], year: int) -> tuple[float, float]:
    """Cost of a proposal scenario in US dollars and as a share of GDP in one baseline year.

    Parameters
    ----------
    scenario_row : dict
        Scenario of the proposal with the cost ``capex_thb_bn`` in billion baht.
    year : int
        Baseline year whose exchange rate and GDP convert the cost.

    Returns
    -------
    capex_usd_bn : float
        Cost in billion US dollars at the exchange rate of ``year``.
    share_pct : float
        Cost in percent of the GDP in US dollars of ``year``.
    """
    capex_usd = float(scenario_row["capex_thb_bn"]) / float(baseline.loc[year, "fx_thb_per_usd"])
    return capex_usd, 100.0 * capex_usd / float(baseline.loc[year, "gdp_usd_bn"])

## Inputs, primary sample and economy groups

The primary sample is the set of treatment episodes whose synthetic control fits the pre-opening years (the flag sample_primary of the effects table). Episodes of one economy are not independent, so the economy code defines the groups of every validation fold and of the cluster bootstrap. The sensitivity samples are the parks-only episodes and the first episode of each economy, both within the primary sample, and all feasible episodes.

In [ ]:
effects = ctx.read_table("case_effects")
placebos = ctx.read_table("case_placebos")
case_features = ctx.read_table("case_features_imputed")
reference = ctx.read_table("reference_features")
weight_table = ctx.read_table("transport_weights").set_index("feature")
panel = panel_module.read_panel(PANEL_PATH)
assert "comparable" in placebos.columns, "case_placebos.csv has no column comparable; run 02_case_effects first"

FEATURES = list(weight_table.index)
lacking_columns = [f for f in FEATURES if f not in case_features.columns or f not in reference.columns]
assert not lacking_columns, f"features without a column in the case features or the reference: {lacking_columns}"

analysis = effects.rename(columns={"capex_pct_gdp": "capex_pct_gdp_observed"}).merge(
    case_features[["case_id", *FEATURES]], on="case_id", how="left", validate="one_to_one"
)
VALUE_COLUMNS = ["att_pp", "att_rel_pct", "placebo_sd_pp", "placebo_sd_rel_pct"]
analysis["usable"] = np.isfinite(analysis[[*VALUE_COLUMNS, *FEATURES]].to_numpy(dtype=float)).all(axis=1)
analysis = analysis.set_index("case_id", drop=False)

SAMPLES = {
    "primary": analysis["sample_primary"].astype(bool) & analysis["usable"],
    "parks_only": analysis["sample_primary"].astype(bool) & analysis["usable"] & analysis["parks_only"].astype(bool),
    "first_episode": analysis["sample_primary"].astype(bool) & analysis["usable"] & analysis["first_episode"].astype(bool),
    "all_feasible": analysis["usable"],
}
primary = analysis[SAMPLES["primary"]]
assert len(primary) >= 4, "the primary sample needs at least four episodes"
GROUPS = primary["economy_group"].to_numpy()

print(f"Feasible episodes: {len(analysis)}; flagged as primary: {int(analysis['sample_primary'].sum())}; primary with complete values: {len(primary)}")
sample_table = pd.DataFrame(
    {
        "episodes": {name: int(mask.sum()) for name, mask in SAMPLES.items()},
        "economies": {name: int(analysis.loc[mask, "economy_group"].nunique()) for name, mask in SAMPLES.items()},
    }
)
print("\nSamples")
print(sample_table.to_string())
per_economy = primary.groupby("economy_group").size()
print(f"\nPrimary sample: {len(primary)} episodes in {per_economy.size} economies")
print("Economies with more than one primary episode:", {k: int(v) for k, v in per_economy[per_economy > 1].items()} or "none")
print("\nFeatures of the cost (weights from transport_weights.csv)")
show(weight_table.loc[FEATURES])

## Standardisation

Every feature is centred at the median of the reference distribution (all economy-years of the reference table) and divided by its median absolute deviation, scaled to match the standard deviation of a normal distribution. A feature that the reference table cannot describe, because fewer than the minimum number of reference rows (printed in the settings) have a value for it, takes its centre and scale from the pooled episodes, which are all feasible episodes with complete features. Standardised values are clipped at the bound printed in the settings, so that no single economy-year dominates the cost. The standardisation is applied to the sources, the validation clouds and the target clouds alike. The table of the largest absolute standardised values before clipping shows which episodes lie far from the reference distribution.

In [ ]:
reference_rows = reference[FEATURES].notna().sum()
LACKING = [f for f in FEATURES if reference_rows[f] < MIN_REFERENCE_ROWS]
SUPPORTED = [f for f in FEATURES if f not in LACKING]
REFERENCE = reference[FEATURES].copy()
for name in LACKING:
    REFERENCE[name] = np.nan
POOLED = analysis.loc[analysis["usable"], FEATURES]
_reference_centre = REFERENCE.median()
CENTRE = _reference_centre.where(_reference_centre.notna(), POOLED.median())
_, _, SCALE = transport.robust_standardise(POOLED, POOLED, reference=REFERENCE, columns=FEATURES, clip=None)

Z_RAW = standardise(analysis[FEATURES], clip=None)
assert np.allclose(Z_RAW.to_numpy(dtype=float), ((analysis[FEATURES] - CENTRE) / SCALE).to_numpy(dtype=float), equal_nan=True), "centre and scale differ from the standardisation"
Z_ALL = standardise(analysis[FEATURES])
Z_SOURCES = Z_ALL.loc[primary.index]
scale_table = pd.DataFrame(
    {
        "reference rows": reference_rows,
        "centre": CENTRE,
        "scale": SCALE,
        "source of centre and scale": ["pooled episodes" if f in LACKING else "reference" for f in FEATURES],
    }
)
show(scale_table)
n_clipped_sources = int((Z_RAW.loc[primary.index].abs() > CLIP).to_numpy().sum())
print(
    f"\nStandardised sources: {Z_SOURCES.shape[0]} episodes by {Z_SOURCES.shape[1]} features; "
    f"{n_clipped_sources} of {Z_SOURCES.size} values were clipped at plus and minus {CLIP:g}"
)

W = {
    "importance": weight_table.loc[FEATURES, "weight_importance"],
    "shrunk": weight_table.loc[FEATURES, "weight_shrunk"],
    "uniform": weight_table.loc[FEATURES, "weight_uniform"],
}
if not float(W["importance"].sum()) > 0.0:
    W["importance"] = W["uniform"].copy()
    print("Every importance weight is zero; the importance weights are replaced by uniform weights.")
Z_RAW_SOURCES = Z_RAW.loc[primary.index]
extreme_table = pd.DataFrame(
    {
        "largest absolute value before clipping": Z_RAW_SOURCES.abs().max(),
        "episode": Z_RAW_SOURCES.abs().idxmax(),
        "values clipped": (Z_RAW_SOURCES.abs() > CLIP).sum(),
    }
)
print("\nLargest absolute standardised value of each feature among the sources")
show(extreme_table, 2)
TAU_PP = primary["att_pp"].to_numpy(dtype=float)
TAU_REL = primary["att_rel_pct"].to_numpy(dtype=float)
SE_PP = primary["placebo_sd_pp"].to_numpy(dtype=float)
SE_REL = primary["placebo_sd_rel_pct"].to_numpy(dtype=float)
print(f"Effects of the primary sample: att_pp from {TAU_PP.min():.3f} to {TAU_PP.max():.3f} percentage points of GDP; att_rel_pct from {TAU_REL.min():.1f} to {TAU_REL.max():.1f} percent")

## Leave-one-economy-out validation

For each primary episode the validation target is the cloud of its own yearly feature vectors in the pre-opening years; the cost share and the integrated-resort flag of the cloud are those of the episode in the imputed feature table. The sources are all primary episodes outside its economy. Six predictors are compared: the transported effect with importance weights (ot_weighted) and with uniform weights (ot_uniform), the plain mean of the other episodes (equal), the mean of the one and of the three nearest episodes (nn1, nn3) and a kernel average (kernel). The regularisation strength is chosen with select_eps inside every fold; when the median pairwise distance of a fold is zero, as happens when the learned weights sit on a feature with few values, select_eps uses the median of the positive distances, and the number of such fold solves is printed. A solve whose scaling iterations do not reach the convergence tolerance within the iteration limit is solved again in the log domain, which converges for small regularisation strengths; the number of such rescued solves is printed and saved next to the number of solves that do not converge. A solve that does not reach the tolerance even after the rescue is not a valid result, so the notebook stops when one occurs.

The importance weights are relearned in every fold from the training episodes alone, with the light configuration of the importance estimator, so that the held-out effect carries no information into the weights of its fold. The weights of the transport plan for the target economy come from notebook 03 and use the full configuration, so the validation judges a cheaper version of the weights than the plan uses, and the RMSE ratios of the folds are not the ratios that the full configuration would give. A fold without enough training economies for the importance estimate uses uniform weights and is counted. A bootstrap over whole economies gives intervals for the ratio of the RMSE of ot_weighted to those of equal and of ot_uniform; the intervals describe the episodes at hand and are not a test of the route rule. The effect in percent of the counterfactual receipts is validated in the same way with the same fold weights, which are learned from the effect in percentage points of GDP.

The validation predicts every episode at its own pre-opening features, so its errors measure how well the transport interpolates among the cases. The transported effect of the target economy is not looked at before these results have been printed, and it is not printed in this notebook at all.

In [ ]:
validation_clouds: dict[str, pd.DataFrame] = {}
filled_entries = 0
clipped_in_clouds = 0
values_in_clouds = 0
for case_id, row in primary.iterrows():
    years = [int(row["opening_year"]) - k for k in range(PRE_YEARS, 0, -1)]
    raw_cloud = cases.economy_feature_cloud(panel, row["iso3"], years)
    for name in ("capex_pct_gdp", "is_integrated_resort"):
        if name in FEATURES:
            raw_cloud[name] = float(analysis.loc[case_id, name])
    filled, n_filled = fill_cloud(raw_cloud, FEATURES, analysis.loc[case_id, FEATURES].astype(float))
    filled_entries += n_filled
    n_clip, n_val = count_clipped(filled)
    clipped_in_clouds += n_clip
    values_in_clouds += n_val
    validation_clouds[case_id] = standardise(filled)
    if ctx.guard:
        ctx.guard()
print(
    f"Validation clouds: {len(validation_clouds)} episodes with {PRE_YEARS} yearly vectors each; {filled_entries} missing entries filled; "
    f"{clipped_in_clouds} of {values_in_clouds} standardised values clipped at plus and minus {CLIP:g}"
)

assert (SE_PP > 0).all() and (SE_REL > 0).all(), "the validation needs strictly positive standard errors"
fold_weights = FoldImportance(primary[FEATURES], TAU_PP, GROUPS, IMPORTANCE_REPEATS, SEED_IMPORTANCE, MIN_FOLD_ECONOMIES, ctx.guard)
loco = transport.loco_validation(
    Z_SOURCES,
    TAU_PP,
    SE_PP,
    fold_weights,
    clouds=validation_clouds,
    eps=None,
    rho_source=RHO_SOURCE,
    a=None,
    seed=SEED_LOCO,
    n_boot=N_BOOT_LOCO,
    groups=GROUPS,
    guard=ctx.guard,
)
fold_table = fold_weights.table()
N_FOLDS = int(fold_table["held_out_group"].nunique())
N_UNIFORM_FOLDS = int(fold_table.loc[~fold_table["fitted"], "held_out_group"].nunique())
print(f"\nImportance relearned in {N_FOLDS - N_UNIFORM_FOLDS} of {N_FOLDS} folds; uniform fallback weights in {N_UNIFORM_FOLDS} folds")
for label, record in fold_weights.records.items():
    if not record["fitted"]:
        print(f"  held-out economy {label}: uniform weights ({record['reason']})")
weights_by_fold = fold_table.pivot(index="held_out_group", columns="feature", values="weight")[FEATURES]
stability = pd.DataFrame(
    {
        "weight on all episodes": W["importance"],
        "mean over folds": weights_by_fold.mean(),
        "smallest in a fold": weights_by_fold.min(),
        "largest in a fold": weights_by_fold.max(),
    }
)
print("\nImportance weights relearned in the folds, compared with the weights learned on all primary episodes")
show(stability)
print(f"Largest weight of a single feature in a fold: {weights_by_fold.max(axis=1).max():.3f}")

print("\nLeave-one-economy-out summary (effects in percentage points of GDP)")
show(loco.summary)
print(
    "The column correlation of the method equal is not informative: the prediction of equal is the mean of the other economies, which falls "
    "as the held-out effect rises, so its correlation with the observed effects is negative by construction and close to minus one when most economies contribute one episode."
)
print("\nRatio of the RMSE of ot_weighted to the comparator, with bootstrap intervals over economies")
show(loco.ratios.drop(columns=["numerator", "denominator"]))
print(loco.ratio_note)
sources_used = loco.table.groupby("method")["n_sources_used"].agg(["min", "max"]).iloc[0]
print(f"\nSources per fold: {int(sources_used['min'])} to {int(sources_used['max'])}; bootstrap resamples: {loco.n_boot}")
print(f"Solves that did not converge: {loco.n_nonconverged}; solves rescued in the log domain: {loco.n_log_domain}; fold solves with the fallback regularisation strength: {loco.n_eps_fallback}")
print(f"Regularisation strength by method (median over the folds of the value selected in each fold): { {k: round(v, 4) for k, v in loco.eps.items()} }; source penalty rho: {loco.rho_source}")

loco_rel = transport.loco_validation(
    Z_SOURCES,
    TAU_REL,
    SE_REL,
    fold_weights,
    clouds=validation_clouds,
    eps=None,
    rho_source=RHO_SOURCE,
    a=None,
    seed=SEED_LOCO,
    n_boot=N_BOOT_LOCO,
    groups=GROUPS,
    guard=ctx.guard,
)
assert len(fold_weights.records) == N_FOLDS, "the second validation must reuse the weights of the folds"
print(f"Effect in percent of the counterfactual receipts: solves that did not converge {loco_rel.n_nonconverged}; solves rescued in the log domain {loco_rel.n_log_domain}; fold solves with the fallback regularisation strength {loco_rel.n_eps_fallback}")
for label, result in (("percentage points of GDP", loco), ("percent of the counterfactual receipts", loco_rel)):
    if result.n_nonconverged > 0:
        raise RuntimeError(
            f"{result.n_nonconverged} Sinkhorn solves of the validation of the effect in {label} did not converge, so the "
            f"validation is not a valid result and the notebook stops. Weights concentrated on one feature give a very small "
            f"regularisation strength; {result.n_eps_fallback} fold solves used the fallback strength. The largest weight of a "
            f"single feature in a fold is {weights_by_fold.max(axis=1).max():.3f}; inspect the fold weights printed above."
        )

print("\nLeave-one-economy-out summary (effects in percent of the counterfactual receipts)")
show(loco_rel.summary)
print("\nRatio of the RMSE of ot_weighted to the comparator for the effect in percent of the counterfactual receipts")
show(loco_rel.ratios.drop(columns=["numerator", "denominator"]))

The leave-one-economy-out table, summary and ratios of the effect in percentage points of GDP are saved with the exact names below, the same three tables of the effect in percent of the counterfactual receipts with the suffix rel, and the importance weights of every fold. The first panel of the figure shows the RMSE of every predictor and the second panel the descriptive bootstrap intervals of the two ratios against the equal-RMSE line and the limit of the route rule.

In [ ]:
saved_loco = [
    ctx.save_table(loco.table, "loco_table"),
    ctx.save_table(loco.summary, "loco_summary", index=True),
    ctx.save_table(loco.ratios, "loco_ratios", index=True),
    ctx.save_table(loco_rel.table, "loco_table_rel"),
    ctx.save_table(loco_rel.summary, "loco_summary_rel", index=True),
    ctx.save_table(loco_rel.ratios, "loco_ratios_rel", index=True),
    ctx.save_table(fold_table, "loco_fold_weights"),
]

RULE_LIMIT = float(impact.DEFAULT_ROUTE_RULES["rmse_ratio_equal"])
METHOD_ORDER = [m for m in ("equal", "nn1", "nn3", "kernel", "ot_uniform", "ot_weighted") if m in loco.summary.index]
fig, (ax_rmse, ax_ratio) = plt.subplots(1, 2, figsize=(11.0, 4.4), gridspec_kw={"width_ratios": [1.1, 1.0]})
rmse = loco.summary["rmse"].reindex(METHOD_ORDER)
positions = np.arange(len(METHOD_ORDER))
bars = ax_rmse.barh(
    positions,
    rmse.to_numpy(),
    color=["0.35" if m == "ot_weighted" else "0.8" for m in METHOD_ORDER],
    edgecolor="0.0",
    hatch=None,
)
for bar, m in zip(bars, METHOD_ORDER):
    if m == "ot_uniform":
        bar.set_hatch("//")
for pos, value in zip(positions, rmse.to_numpy()):
    ax_rmse.text(value, pos, f" {value:.3f}", va="center", ha="left", fontsize=8)
ax_rmse.set_yticks(positions)
ax_rmse.set_yticklabels(METHOD_ORDER)
ax_rmse.set_xlim(0, float(rmse.max()) * 1.18)
ax_rmse.set_xlabel("Leave-one-economy-out RMSE (percentage points of GDP)")
ax_rmse.set_ylabel("Method")
ax_rmse.set_title("Prediction error by method")
ax_rmse.grid(axis="y", visible=False)

rows = list(loco.ratios.index)
ypos = np.arange(len(rows))[::-1]
for y, name in zip(ypos, rows):
    r = loco.ratios.loc[name]
    ax_ratio.hlines(y, r["lo95"], r["hi95"], color="0.45", linewidth=1.2, label="95 percent interval" if y == ypos[0] else None)
    ax_ratio.hlines(y, r["lo80"], r["hi80"], color="0.0", linewidth=4.0, label="80 percent interval" if y == ypos[0] else None)
    ax_ratio.plot(r["ratio"], y, marker="o", markerfacecolor="white", markeredgecolor="0.0", markersize=7, linestyle="none", label="Sample ratio" if y == ypos[0] else None)
ax_ratio.axvline(1.0, color="0.0", linestyle="-", linewidth=1.0, label="Equal RMSE (ratio 1)")
ax_ratio.axvline(RULE_LIMIT, color="0.3", linestyle="--", linewidth=1.0, label=f"Limit of the route rule ({RULE_LIMIT:g})")
ax_ratio.set_yticks(ypos)
ax_ratio.set_yticklabels(rows)
ax_ratio.set_ylim(-0.7, len(rows) - 0.3)
ax_ratio.set_xlabel("RMSE of ot_weighted divided by RMSE of the comparator (ratio)")
ax_ratio.set_ylabel("Comparison")
ax_ratio.set_title("Bootstrap intervals over economies (descriptive)")
ax_ratio.legend(loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=8)
ax_ratio.grid(axis="y", visible=False)
fig.suptitle("Leave-one-economy-out validation of the transport rule", y=1.02)
fig.tight_layout()
ctx.save_figure(fig, "ot_loco_validation.png")

for name in loco.ratios.index:
    r = loco.ratios.loc[name]
    print(
        f"{name}: ratio {r['ratio']:.3f}, 80 percent interval {r['lo80']:.3f} to {r['hi80']:.3f}, "
        f"95 percent interval {r['lo95']:.3f} to {r['hi95']:.3f}; share of resamples below one {r['share_below_one']:.2f}; limit of the route rule {RULE_LIMIT:g}"
    )

## Target economy

This section is the single description of the target economy. The next notebook reads it from thailand_transport.json and thailand_target_baseline.csv and does not compute it again.

The effects of the cases are measured on the receipts series of the panel, which covers travel and passenger transport, whereas the baseline file holds the balance-of-payments travel credits, which are smaller. The baseline is therefore put on the basis of the panel with impact.to_panel_basis: a year in which the panel has a value takes the panel value, and any other year takes the baseline value times the mean ratio of the panel receipts to the baseline receipts over the ratio years printed in the settings. The receipts share of GDP is recomputed from the receipts on the basis of the panel.

The proposal states the cost in baht. The cost share of GDP is the cost in US dollars at the exchange rate of one baseline year divided by the GDP in US dollars of that year, which equals the cost in baht divided by the GDP in baht. The primary reading uses the latest baseline year that has a GDP and an exchange rate, because the cost is stated in current prices; the alternative reading uses the year printed in the settings. Every target cloud holds the cost share constant in all its rows, as the clouds of the validation episodes do. The primary cloud uses the primary scenario and the pre-pandemic years. The sensitivity clouds use the alternative cost year, each of the other scenarios, and the integrated-resort flag set to one.

In [ ]:
baseline_all = impact.load_thailand_baseline(ctx.baseline_path)
COST_YEAR = int(baseline_all.dropna(subset=["gdp_usd_bn", "fx_thb_per_usd"]).index.max())
baseline_raw = impact.load_thailand_baseline(ctx.baseline_path, years=sorted({COST_YEAR_ALTERNATIVE, COST_YEAR}))
panel_receipts = panel.loc[panel["iso3"] == ctx.target_iso3].set_index("year")["receipts_usd"].astype(float) / 1.0e9
baseline = impact.to_panel_basis(baseline_raw, panel_receipts, ratio_years=RATIO_YEARS)
basis_table = baseline[["receipts_usd_bn_baseline", "receipts_usd_bn", "receipts_basis", "basis_ratio", "receipts_pct_gdp"]]

window_years = list(TARGET_YEARS[-RECEIPTS_WINDOW:])
window_share = baseline.reindex(window_years)["receipts_pct_gdp"]
assert window_share.notna().all(), f"the receipts share of GDP is missing in {[int(y) for y in window_share.index[window_share.isna()]]}"
TARGET_RECEIPTS_SHARE = float(window_share.mean())

scenario = ctx.primary_scenario()
scenario_rows = []
for entry in ctx.proposal["scenarios"]:
    usd_cost, share_cost = cost_share(entry, COST_YEAR)
    _, share_alternative = cost_share(entry, COST_YEAR_ALTERNATIVE)
    scenario_rows.append(
        {
            "scenario": entry["name"],
            "primary": entry["name"] == scenario["name"],
            "capex_thb_bn": float(entry["capex_thb_bn"]),
            "capex_usd_bn": usd_cost,
            "capex_pct_gdp": share_cost,
            "capex_pct_gdp_alternative": share_alternative,
        }
    )
scenario_table = pd.DataFrame(scenario_rows).set_index("scenario", drop=False)
PRIMARY_ROW = scenario_table.loc[scenario["name"]]
FX_COST = float(baseline.loc[COST_YEAR, "fx_thb_per_usd"])
GDP_COST = float(baseline.loc[COST_YEAR, "gdp_usd_bn"])

print(f"Target economy: {ctx.target_name}; proposal status: {ctx.proposal.get('status', 'not stated')}")
print("\nReceipts of the target on the basis of the panel (US$ billion; the baseline column holds the balance-of-payments travel credits)")
show(basis_table, 3)
print(
    f"\nCost year {COST_YEAR}: {FX_COST:.3f} THB per US$ and a GDP of US$ {GDP_COST:.1f} billion; alternative cost year {COST_YEAR_ALTERNATIVE}: "
    f"{float(baseline.loc[COST_YEAR_ALTERNATIVE, 'fx_thb_per_usd']):.3f} THB per US$ and a GDP of US$ {float(baseline.loc[COST_YEAR_ALTERNATIVE, 'gdp_usd_bn']):.1f} billion"
)
print(f"Primary scenario {scenario['name']}: THB {PRIMARY_ROW['capex_thb_bn']:g} billion = US$ {PRIMARY_ROW['capex_usd_bn']:.3f} billion = {PRIMARY_ROW['capex_pct_gdp']:.3f} percent of GDP")
print("\nScenarios of the proposal (cost share of GDP in percent)")
show(scenario_table.drop(columns=["scenario"]), 3)
print(f"\nReceipts as a share of GDP on the basis of the panel, mean of {window_years[0]} to {window_years[-1]}: {TARGET_RECEIPTS_SHARE:.3f} percent")

CLOUD_SPECS: dict[str, dict[str, object]] = {
    "primary": {"scenario": scenario["name"], "share": float(PRIMARY_ROW["capex_pct_gdp"]), "resort": 0.0},
    f"cost_year_{COST_YEAR_ALTERNATIVE}": {"scenario": scenario["name"], "share": float(PRIMARY_ROW["capex_pct_gdp_alternative"]), "resort": 0.0},
}
for name, row in scenario_table.iterrows():
    if not row["primary"]:
        CLOUD_SPECS[f"scenario_{name}"] = {"scenario": name, "share": float(row["capex_pct_gdp"]), "resort": 0.0}
CLOUD_SPECS["integrated_resort"] = {"scenario": scenario["name"], "share": float(PRIMARY_ROW["capex_pct_gdp"]), "resort": 1.0}
SCENARIO_CLOUDS = {spec["scenario"]: name for name, spec in CLOUD_SPECS.items() if name == "primary" or name.startswith("scenario_")}

RAW_CLOUDS: dict[str, pd.DataFrame] = {}
FILLED_CLOUDS: dict[str, pd.DataFrame] = {}
Z_CLOUDS: dict[str, pd.DataFrame] = {}
for name, spec in CLOUD_SPECS.items():
    raw = cases.economy_feature_cloud(
        panel, ctx.target_iso3, TARGET_YEARS, capex_pct_gdp=spec["share"], is_integrated_resort=spec["resort"]
    ).set_index("year")
    filled, n_filled = fill_cloud(raw, FEATURES, CENTRE)
    RAW_CLOUDS[name] = raw
    FILLED_CLOUDS[name] = filled
    Z_CLOUDS[name] = standardise(filled)
    n_clip, n_val = count_clipped(filled)
    print(
        f"Cloud {name}: years {list(raw.index)}, constant cost share {spec['share']:.3f} percent of GDP, {n_filled} missing entries filled, "
        f"{n_clip} of {n_val} standardised values clipped"
    )

Z_TARGET = Z_CLOUDS["primary"]
print("\nPrimary target cloud, raw features")
show(RAW_CLOUDS["primary"][[f for f in cases.FEATURE_COLUMNS]], 3)
print("\nPrimary target cloud, standardised features used in the cost")
show(Z_TARGET, 3)

## Transport plan

The plan uses importance weights, the source penalty RHO_SOURCE and the regularisation strength select_eps of the primary sources. Each target point receives its mass exactly, and sources may send more or less than their own mass. The transported effect of a target point is the mass-weighted mean of the source effects that arrive there, and the target effect is the average over the target points. The transported effects are computed and kept for the next notebook; this section prints the properties of the plan only, and the table of source usage leaves out the effects of the source episodes.

In [ ]:
main = solve_transport(Z_SOURCES, Z_TARGET, W["importance"], RHO_SOURCE)
plan = main["plan"]
EPS_MAIN = float(main["eps"])
B_TARGET = main["b"]
assert plan.converged, f"the Sinkhorn iterations did not converge (marginal error relative to the total mass {plan.marginal_error:.2e})"
PLAN_LOG_DOMAIN = bool(plan.log_domain)

usage = transport.source_usage(plan)
usage_by_economy = usage.groupby(GROUPS).sum()
ess_episodes = transport.effective_sample_size(usage.to_numpy())
ess_economies = transport.effective_sample_size(usage_by_economy.to_numpy())
effect_pp = transport.target_effect(plan, TAU_PP, B_TARGET)
effect_rel = transport.target_effect(plan, TAU_REL, B_TARGET)
by_year = transport.transported_effects(plan, TAU_PP).rename(columns={"effect": "att_pp"})
by_year["att_rel_pct"] = transport.transported_effects(plan, TAU_REL)["effect"]

print(f"Regularisation strength eps = {EPS_MAIN:.4f}; source penalty rho = {RHO_SOURCE}; iterations {plan.n_iter}; marginal error relative to the total mass {plan.marginal_error:.1e}; solved in the log domain: {PLAN_LOG_DOMAIN}")
print(f"Transported mass {plan.mass:.4f} (target mass {B_TARGET.sum():.4f}); mass sent per source from {usage.min():.4f} to {usage.max():.4f} (own mass {1.0 / len(usage):.4f})")
print(f"Effective sample size: {ess_episodes:.2f} episodes of {len(usage)}; {ess_economies:.2f} economies of {usage_by_economy.size}")

usage_table = pd.DataFrame(
    {
        "case_id": usage.index,
        "iso3": primary["iso3"].to_numpy(),
        "opening_year": primary["opening_year"].to_numpy(),
        "att_pp": TAU_PP,
        "att_rel_pct": TAU_REL,
        "mass_sent": usage.to_numpy(),
        "share_of_mass": usage.to_numpy() / usage.sum(),
    }
).sort_values("mass_sent", ascending=False)
print("\nSource usage (the effects of the source episodes are in transport_usage.csv and are not shown here)")
show(usage_table.drop(columns=["att_pp", "att_rel_pct"]).set_index("case_id"), 4)

plan_table = pd.DataFrame(plan.pi, index=pd.Index(Z_SOURCES.index, name="case_id"), columns=[str(y) for y in Z_TARGET.index])
saved_plan = ctx.save_table(plan_table, "transport_plan", index=True)
saved_usage = ctx.save_table(usage_table, "transport_usage")

## Support of the target

A barycentric transport cannot exceed the largest effect among its sources, so a target that is costlier, larger or more concentrated than every case is an extrapolation that the transport cannot make. Two checks describe how far the target lies from what the sources can represent. The first is the effective number of sources behind the target, which is computed from the mass that each source sends under the plan; it falls towards one when a single episode carries the target. The second is a range test on the raw features: a feature is outside when the target lies beyond the range of the sources that send mass, widened by the margin printed in the settings, and the target is supported only when no feature with a weight of at least the limit printed in the settings is outside. The range test uses the raw values, so the clipping of the standardised values does not hide a target that is far out, and it covers the cost share and the integrated-resort flag, which the overlap test below can describe only through the sources. The result is saved with the transported effects and the route rule of the next notebook reads it as its fourth criterion.

In [ ]:
REFERENCE_FULL = REFERENCE.copy()
assert len(POOLED) <= len(REFERENCE), "the reference table is shorter than the pooled episodes"
for name in LACKING:
    REFERENCE_FULL[name] = pd.Series(POOLED[name].to_numpy(dtype=float), index=REFERENCE.index[: len(POOLED)]).reindex(REFERENCE.index)

support = transport.target_support(
    primary[FEATURES],
    FILLED_CLOUDS["primary"][FEATURES],
    W["importance"],
    plan,
    reference=REFERENCE_FULL,
    min_ess=MIN_ESS,
    min_weight=MIN_WEIGHT,
    range_tolerance=RANGE_TOLERANCE,
    clip=CLIP,
)
print(f"Effective number of sources behind the target: {support['ess']:.2f} (limit {MIN_ESS:g}); sources that send mass: {support['n_sources']} of {len(primary)}; largest share of the mass of one source: {100 * support['max_source_share']:.1f} percent")
print("\nRange of the sources that send mass against the target (raw features; the target value is the median of the cloud)")
show(support["features"], 3)
print(f"\nShare of the importance weight on features outside the band: {support['weighted_outside_share']:.3f}")
print(f"Effective number of sources criterion met: {support['ess_ok']}; range criterion met: {support['range_ok']}")
print(f"Target supported: {support['supported']}")
for reason in support["reasons"]:
    print(f"  {reason}")
if support["supported"]:
    print("The support check reports no reason against the transport.")

## Overlap between the target and the sources

The statistic is the mean over the target points of the weighted squared distance to the nearest source. Its null distribution comes from clouds of the same size drawn at random from the standardised reference economy-years. A small p-value means that the target lies farther from the sources than most random clouds do.

The test uses every feature of the cost, including the cost share and the integrated-resort flag, which carry weight in the cost. The reference table has no values for features that describe an episode and not an economy-year. Each reference row of the null clouds therefore takes the value of such a feature from a source episode chosen at random, so that the random clouds have the distribution of those features among the cases. A target whose cost share lies far beyond the cost shares of the cases then looks far from the sources in the test, as it does in the support check above.

In [ ]:
test_features = list(FEATURES)
w_test = W["importance"][test_features]
if float(w_test.sum()) <= 0.0:
    w_test = W["uniform"][test_features]
    print("The importance weights are zero on the features of the test; uniform weights are used in the test.")
pool_raw = reference.set_index(["iso3", "year"])[SUPPORTED].dropna()
pool_rng = np.random.default_rng(SEED_PERM)
for name in LACKING:
    pool_raw[name] = pool_rng.choice(primary[name].to_numpy(dtype=float), size=len(pool_raw))
pool_z = standardise(pool_raw[test_features], test_features)
overlap = transport.overlap_permutation_test(
    Z_SOURCES[test_features],
    Z_TARGET[test_features],
    w_test,
    a=None,
    b=None,
    n_perm=N_PERM,
    seed=SEED_PERM,
    pool=pool_z,
)
null_stat = overlap["null"]
print(f"Features in the test: {test_features}")
print(f"Features whose values in the random clouds come from randomly chosen source episodes (no reference values): {LACKING or 'none'}")
print(f"Reference pool: {len(pool_z)} economy-years; permutations: {overlap['n_perm']}")
print(f"Statistic (mean weighted squared distance to the nearest source): {overlap['statistic']:.4f}")
print(f"Null distribution: median {np.median(null_stat):.4f}, 95th percentile {np.quantile(null_stat, 0.95):.4f}")
print(f"Share of random clouds at least as far from the sources: p = {overlap['p_value']:.4f}")

## Sensitivity of the transport plan

The plan is recomputed for every combination of target cloud, weight vector and source penalty, each with its own regularisation strength from select_eps, and for the sensitivity samples with the base configuration. The table below shows the properties of each plan. The transported effects of every configuration are saved in transport_sensitivity.csv and are shown in the next notebook after the route decision.

In [ ]:
sensitivity_rows = []
for cloud_name, Z_cloud in Z_CLOUDS.items():
    for weight_name, weights in W.items():
        for rho in (RHO_SOURCE, *RHO_SENSITIVITY):
            run = solve_transport(Z_SOURCES, Z_cloud, weights, rho)
            plan_s = run["plan"]
            sensitivity_rows.append(
                {
                    "sample": "primary",
                    "target_cloud": cloud_name,
                    "weights": weight_name,
                    "rho_source": rho,
                    "n_sources": len(Z_SOURCES),
                    "eps": run["eps"],
                    "att_pp": transport.target_effect(plan_s, TAU_PP, run["b"]),
                    "att_rel_pct": transport.target_effect(plan_s, TAU_REL, run["b"]),
                    "ess": transport.effective_sample_size(transport.source_usage(plan_s).to_numpy()),
                    "mass": plan_s.mass,
                    "converged": bool(plan_s.converged),
                    "log_domain": bool(plan_s.log_domain),
                }
            )
        if ctx.guard:
            ctx.guard()
for sample_name, mask in SAMPLES.items():
    if sample_name == "primary":
        continue
    members = analysis[mask]
    run = solve_transport(Z_ALL.loc[members.index], Z_TARGET, W["importance"], RHO_SOURCE)
    plan_s = run["plan"]
    sensitivity_rows.append(
        {
            "sample": sample_name,
            "target_cloud": "primary",
            "weights": "importance",
            "rho_source": RHO_SOURCE,
            "n_sources": len(members),
            "eps": run["eps"],
            "att_pp": transport.target_effect(plan_s, members["att_pp"].to_numpy(dtype=float), run["b"]),
            "att_rel_pct": transport.target_effect(plan_s, members["att_rel_pct"].to_numpy(dtype=float), run["b"]),
            "ess": transport.effective_sample_size(transport.source_usage(plan_s).to_numpy()),
            "mass": plan_s.mass,
            "converged": bool(plan_s.converged),
            "log_domain": bool(plan_s.log_domain),
        }
    )
sensitivity = pd.DataFrame(sensitivity_rows)
show(sensitivity[["sample", "target_cloud", "weights", "rho_source", "n_sources", "eps", "ess", "mass", "converged", "log_domain"]], 4)
saved_sensitivity = ctx.save_table(sensitivity, "transport_sensitivity")
print(f"\nConfigurations: {len(sensitivity)}; configurations that did not converge: {int((~sensitivity['converged']).sum())}; configurations solved in the log domain: {int(sensitivity['log_domain'].sum())}")

## Sampling uncertainty and predictive intervals of the transported effect

The module produces three different intervals for the transported effect and each answers a different question.

1. The bootstrap interval of the transported average effect resamples whole economies with replacement, adds normal measurement noise with the placebo standard deviation of each effect, recomputes the plan and the target effect. Its draws describe the sampling uncertainty of the transported average effect. They do not describe the effect that one new study would observe. The same seed is used for both outcomes, so the replicates of the effect in percentage points of GDP and in percent of the counterfactual use the same resampled economies.
2. The bootstrap predictive draws add to each replicate the deconvolved between-episode dispersion and the measurement noise of one episode. They are kept as a sensitivity and have no coverage guarantee.
3. The conformal interval is the predictive interval for the effect that a study of the target economy would observe. It is built from the leave-one-economy-out prediction errors of the validated predictor by split conformal prediction: with n errors it is the transported effect plus and minus the q-th smallest absolute error, where q is the level times n plus one, rounded up to an integer. If the error at the target is exchangeable with the validation errors, its coverage is at least the guaranteed level printed below. The coverage guarantee needs at least as many errors as the status line states, and it is approximate when an economy contributes several episodes, because errors of one economy are dependent. The interval is validated for targets that resemble the cases, as checked in the support section above, because the validation errors are errors at the cases' own features.

The predictive draws that are saved resample the validation errors by economy and are rescaled about the transported effect so that their symmetric interval at the stated level equals the conformal interval. The percentiles of those draws are descriptive; only the conformal interval carries the guarantee. The effect in percent of the counterfactual receipts uses the errors of its own validation, and the common seed pairs the two outcomes. The estimates and the intervals are saved and are not printed here.

In [ ]:
LOCO_PRED_PP = loco.table[loco.table["method"] == PREDICTIVE_METHOD]
LOCO_PRED_REL = loco_rel.table[loco_rel.table["method"] == PREDICTIVE_METHOD]


def predictive_pair(estimate_pp: float, estimate_rel: float) -> tuple[dict[str, object], dict[str, object]]:
    """Predictive draws and conformal intervals of both outcomes at one transported estimate.

    Parameters
    ----------
    estimate_pp : float
        Transported effect in percentage points of GDP.
    estimate_rel : float
        Transported effect in percent of the counterfactual receipts.

    Returns
    -------
    tuple of dict
        The outputs of ``transport.loco_predictive_draws`` for the effect in percentage points of GDP and for the
        effect in percent of the counterfactual receipts, with the level ``CONFORMAL_LEVEL`` and the common seed.
    """
    pp = transport.loco_predictive_draws(LOCO_PRED_PP, estimate_pp, n_draws=N_BOOT_TRANSPORT, seed=SEED_LOCO, level=CONFORMAL_LEVEL)
    rel = transport.loco_predictive_draws(LOCO_PRED_REL, estimate_rel, n_draws=N_BOOT_TRANSPORT, seed=SEED_LOCO, level=CONFORMAL_LEVEL)
    return pp, rel


boot_pp = transport.bootstrap_transport(
    Z_SOURCES, Z_TARGET, W["importance"], TAU_PP, SE_PP, a=None, b=None, eps=EPS_MAIN, rho_source=RHO_SOURCE,
    n_boot=N_BOOT_TRANSPORT, seed=SEED_BOOT, groups=GROUPS, guard=ctx.guard,
)
boot_rel = transport.bootstrap_transport(
    Z_SOURCES, Z_TARGET, W["importance"], TAU_REL, SE_REL, a=None, b=None, eps=EPS_MAIN, rho_source=RHO_SOURCE,
    n_boot=N_BOOT_TRANSPORT, seed=SEED_BOOT, groups=GROUPS, guard=ctx.guard,
)
pred_pp, pred_rel = predictive_pair(effect_pp, effect_rel)
for label, boot, pred, unit in (
    ("att_pp", boot_pp, pred_pp, "percentage points of GDP"),
    ("att_rel_pct", boot_rel, pred_rel, "percent of the counterfactual receipts"),
):
    interval = pred["interval"]
    print(
        f"{label} ({unit}): deconvolved between-episode sd {boot['between_sd']:.4f}; measurement sd {boot['measurement_sd']:.4f}; "
        f"bootstrap replicates {boot['n_boot']}, non-converged solves {boot['n_nonconverged']}, solves rescued in the log domain {boot['n_log_domain']}"
    )
    print(
        f"  conformal interval at level {interval['level']:g}: {interval['n']} validation errors in {pred['n_groups']} economies, order statistic {interval['rank']}, "
        f"guaranteed coverage {interval['guaranteed_level']:.3f}, half width {interval['half_width']:.4f} {unit}; status: {interval['status']}"
    )
    print(f"  predictive draws rescaled to the conformal interval: {pred['calibrated']} (scale {pred['scale']:.3f})")

predictive = pd.DataFrame(
    {
        "draw": np.arange(N_BOOT_TRANSPORT),
        "predictive_pp": pred_pp["draws"],
        "predictive_rel_pct": pred_rel["draws"],
        "transport_pp": boot_pp["draws"],
        "transport_rel_pct": boot_rel["draws"],
    }
)
saved_draws = ctx.save_table(predictive, "thailand_transport_draws")

# Scenarios of the proposal: plan, support, transported effect and predictive draws of every scenario
SCENARIO_RESULTS: dict[str, dict[str, object]] = {}
scenario_draw_tables = []
for scenario_name, cloud_name in SCENARIO_CLOUDS.items():
    if cloud_name == "primary":
        run_plan, support_s, est_pp, est_rel, pair = plan, support, effect_pp, effect_rel, (pred_pp, pred_rel)
    else:
        run = solve_transport(Z_SOURCES, Z_CLOUDS[cloud_name], W["importance"], RHO_SOURCE)
        run_plan = run["plan"]
        assert run_plan.converged, f"the plan of the scenario {scenario_name} did not converge"
        support_s = transport.target_support(
            primary[FEATURES], FILLED_CLOUDS[cloud_name][FEATURES], W["importance"], run_plan,
            reference=REFERENCE_FULL, min_ess=MIN_ESS, min_weight=MIN_WEIGHT, range_tolerance=RANGE_TOLERANCE, clip=CLIP,
        )
        est_pp = transport.target_effect(run_plan, TAU_PP, run["b"])
        est_rel = transport.target_effect(run_plan, TAU_REL, run["b"])
        pair = predictive_pair(est_pp, est_rel)
    row = scenario_table.loc[scenario_name]
    SCENARIO_RESULTS[scenario_name] = {
        "primary": bool(row["primary"]),
        "capex_thb_bn": float(row["capex_thb_bn"]),
        "capex_usd_bn": float(row["capex_usd_bn"]),
        "capex_pct_gdp": float(row["capex_pct_gdp"]),
        "support": support_record(support_s),
        "att_pp": {"estimate": float(est_pp), "conformal_interval": interval_record(pair[0]["interval"]), "predictive_percentiles": quantile_dict(pair[0]["draws"])},
        "att_rel_pct": {"estimate": float(est_rel), "conformal_interval": interval_record(pair[1]["interval"]), "predictive_percentiles": quantile_dict(pair[1]["draws"])},
    }
    scenario_draw_tables.append(
        pd.DataFrame(
            {
                "scenario": scenario_name,
                "draw": np.arange(N_BOOT_TRANSPORT),
                "predictive_pp": pair[0]["draws"],
                "predictive_rel_pct": pair[1]["draws"],
            }
        )
    )
scenario_overview = pd.DataFrame(
    {
        "capex_pct_gdp": {k: v["capex_pct_gdp"] for k, v in SCENARIO_RESULTS.items()},
        "ess": {k: v["support"]["ess"] for k, v in SCENARIO_RESULTS.items()},
        "max_source_share": {k: v["support"]["max_source_share"] for k, v in SCENARIO_RESULTS.items()},
        "weighted_outside_share": {k: v["support"]["weighted_outside_share"] for k, v in SCENARIO_RESULTS.items()},
        "supported": {k: v["support"]["supported"] for k, v in SCENARIO_RESULTS.items()},
    }
)
print("\nSupport of the target by scenario of the proposal")
show(scenario_overview, 3)
saved_scenario_draws = ctx.save_table(pd.concat(scenario_draw_tables, ignore_index=True), "thailand_scenario_draws")

## Transported placebo noise

Each source carries the effects that its donors would have received in the placebo-in-space design. Only the placebos whose pre-opening fit is comparable to the fit of the case enter, which is the rule that gives the standard error and the rank p-value of the case in notebook 02; a placebo that fits its own pre-opening years much worse than the donors typically do would otherwise add noise that the case itself does not have. The placebo draws of the sources are pooled with probabilities proportional to the mass that the sources send under the plan. The pooled distribution is the null distribution of the effect at the target: what the method would report where nothing happened. Null draws for the two outcomes are sampled independently from their own pooled distributions. The null of the effect in percentage points of GDP belongs to the economies of the sources; the next notebook puts it on the receipts share of the target before it enters the detection analysis.

In [ ]:
comparable_mask = placebos["comparable"].astype(bool)
placebo_groups = {case_id: frame for case_id, frame in placebos[comparable_mask].groupby("case_id")}
missing_placebos = [c for c in Z_SOURCES.index if c not in placebo_groups]
assert not missing_placebos, f"no comparable placebo rows for {missing_placebos}"
draws_pp = {c: placebo_groups[c]["placebo_att_pp"].to_numpy(dtype=float) for c in Z_SOURCES.index}
draws_rel = {c: placebo_groups[c]["placebo_att_rel_pct"].to_numpy(dtype=float) for c in Z_SOURCES.index}
n_placebos_all = int(placebos["case_id"].isin(Z_SOURCES.index).sum())
n_placebos_comparable = int(sum(len(v) for v in draws_pp.values()))
values_pp, probs_pp = transport.transported_mixture(plan, draws_pp)
values_rel, probs_rel = transport.transported_mixture(plan, draws_rel)
null_quantiles = {
    "att_pp": dict(zip((f"p{round(100 * p):g}" for p in PROBS), transport.weighted_quantile(values_pp, probs_pp, list(PROBS)))),
    "att_rel_pct": dict(zip((f"p{round(100 * p):g}" for p in PROBS), transport.weighted_quantile(values_rel, probs_rel, list(PROBS)))),
}
rng_null = np.random.default_rng(SEED_NULL)
null_pp = rng_null.choice(values_pp, size=N_NULL, p=probs_pp)
null_rel = rng_null.choice(values_rel, size=N_NULL, p=probs_rel)
null_table = pd.DataFrame({"null_pp": null_pp, "null_rel_pct": null_rel})
saved_null = ctx.save_table(null_table, "thailand_null_draws")
print(f"Placebos of the {len(Z_SOURCES)} sources: {n_placebos_comparable} comparable of {n_placebos_all}; comparable placebos per source from {min(len(v) for v in draws_pp.values())} to {max(len(v) for v in draws_pp.values())}")
print(f"Placebo draws pooled: {len(values_pp)} for the effect in percentage points of GDP, {len(values_rel)} for the relative effect")
print("Transported placebo distribution, percentiles 5, 25, 50, 75, 95")
print("  percentage points of GDP:", np.round(list(null_quantiles["att_pp"].values()), 4))
print("  percent of the counterfactual:", np.round(list(null_quantiles["att_rel_pct"].values()), 3))
print(f"Standard deviation of the null draws: {null_pp.std(ddof=1):.4f} percentage points of GDP; {null_rel.std(ddof=1):.3f} percent")

## Figures

The first figure is the transport plan as a heat map of the episodes, sorted by the mass they send, against the target years. The second figure places the sources and the target cloud in the plane of the first two principal components of the standardised features after each feature has been multiplied by the square root of its cost weight, so that distances in the plane approximate the ground cost of the transport; the loadings are printed above it. The third figure, saved in the validation section, shows the leave-one-economy-out errors. The fourth figure shows, for every feature, the range of the sources that send mass and the position of the target, which is the range test of the support check; its horizontal axis is linear within the clip bound and logarithmic beyond it. The figures show the plan and the target description; the figures of the transported effect are in the next notebook.

In [ ]:
order = usage.sort_values(ascending=False).index
heat = plan_table.loc[order]
label_size = 8 if len(order) <= 30 else 6
fig, ax = plt.subplots(figsize=(7.2, min(0.34 * len(order) + 2.2, 18.0)))
image = ax.imshow(heat.to_numpy(), cmap="Greys", aspect="auto", vmin=0.0)
ax.set_xticks(np.arange(heat.shape[1]))
ax.set_xticklabels(heat.columns)
ax.set_yticks(np.arange(heat.shape[0]))
ax.set_yticklabels(heat.index, fontsize=label_size)
threshold = 0.5 * float(heat.to_numpy().max())
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        value = heat.to_numpy()[i, j]
        ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=label_size - 1.5, color="white" if value > threshold else "black")
ax.set_xlabel("Target year of the feature vector")
ax.set_ylabel("Source episode (sorted by mass sent)")
ax.set_title("Transport plan: mass sent from each episode to each target year")
ax.grid(False)
colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label("Transported mass")
fig.tight_layout()
figure_path = ctx.save_figure(fig, "ot_plan_heatmap.png")

In [ ]:
geometry = np.sqrt(len(FEATURES) * W["importance"].to_numpy() / float(W["importance"].sum()))
stack = np.vstack([Z_SOURCES.to_numpy(), Z_TARGET.to_numpy()]) * geometry
centre, axes, share = principal_axes(stack)
source_scores = (stack[: len(Z_SOURCES)] - centre) @ axes.T
target_scores = (stack[len(Z_SOURCES) :] - centre) @ axes.T
loadings = pd.DataFrame(axes.T, index=FEATURES, columns=["PC1 loading", "PC2 loading"])
loadings["importance weight"] = W["importance"].to_numpy()
print(
    f"Each feature is multiplied by the square root of its cost weight, so that distances in the plane approximate the ground cost; "
    f"the standardised values are those of the cost, clipped at plus and minus {CLIP:g}"
)
print(f"Share of variance: PC1 {100 * share[0]:.1f} percent, PC2 {100 * share[1]:.1f} percent")
show(loadings, 3)

sent = usage.loc[Z_SOURCES.index].to_numpy()
sizes = 30.0 + 500.0 * sent / sent.max()
fig, ax = plt.subplots(figsize=(7.4, 5.8))
ax.scatter(
    source_scores[:, 0], source_scores[:, 1], s=sizes, facecolor="0.7", edgecolor="0.0", linewidth=0.8,
    label="Source episode (marker area proportional to mass sent)", zorder=3,
)
ax.plot(
    target_scores[:, 0], target_scores[:, 1], marker="D", markersize=7, color="0.0", linestyle="-", linewidth=1.0,
    label="Target economy, yearly feature vectors", zorder=4,
)
ax.set_xlabel(f"First principal component ({100 * share[0]:.0f} percent of the variance; weighted standardised units)")
ax.set_ylabel(f"Second principal component ({100 * share[1]:.0f} percent of the variance; weighted standardised units)")
ax.set_title("Sources and target cloud in the cost geometry")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8)
fig.tight_layout()

renderer = fig.canvas.get_renderer()
pixels_per_point = fig.dpi / 72.0
occupied = []
for x_pos, y_pos, area in zip(source_scores[:, 0], source_scores[:, 1], sizes):
    cx, cy = ax.transData.transform((x_pos, y_pos))
    half = float(np.sqrt(area / np.pi)) * pixels_per_point
    occupied.append(Bbox.from_extents(cx - half, cy - half, cx + half, cy + half))
half = 4.5 * pixels_per_point
for x_pos, y_pos in zip(target_scores[:, 0], target_scores[:, 1]):
    cx, cy = ax.transData.transform((x_pos, y_pos))
    occupied.append(Bbox.from_extents(cx - half, cy - half, cx + half, cy + half))
for a, b in zip(target_scores[:-1], target_scores[1:]):
    (ax0, ay0), (ax1, ay1) = ax.transData.transform(a), ax.transData.transform(b)
    occupied.append(Bbox.from_extents(min(ax0, ax1), min(ay0, ay1) - 1.0, max(ax0, ax1), max(ay0, ay1) + 1.0))
axes_box = ax.get_window_extent(renderer)


def place_label(text: str, point: tuple[float, float], radius: float, bold: bool = False, prefer: int = 1) -> None:
    """Annotate a point with the first of several offsets whose text box is free.

    Parameters
    ----------
    text : str
        Label to draw.
    point : tuple of float
        Data coordinates of the labelled point.
    radius : float
        Radius of the marker in points; the label starts beyond it.
    bold : bool
        Whether the label is drawn in bold face.
    prefer : int
        1 to try the positions to the right of the point first, -1 to try those to the left first.
    """
    gap = radius + 6.0
    options = []
    for tier in (0.0, 1.0):
        for sign in (prefer, -prefer):
            for dy in (6.0, -12.0):
                options.append((sign * gap, dy + np.sign(dy) * 12.0 * tier, "left" if sign > 0 else "right"))
    arrow = {"arrowstyle": "-", "color": "0.4", "linewidth": 0.6, "shrinkA": 0.0, "shrinkB": radius}
    weight = "bold" if bold else "normal"
    margin = 3.0 * pixels_per_point
    for dx, dy, side in options:
        note = ax.annotate(text, point, xytext=(dx, dy), textcoords="offset points", ha=side, fontsize=7, fontweight=weight, arrowprops=arrow)
        note.get_window_extent(renderer)
        box = Text.get_window_extent(note, renderer)
        padded = Bbox.from_extents(box.x0 - margin, box.y0 - margin, box.x1 + margin, box.y1 + margin)
        inside = axes_box.contains(box.x0, box.y0) and axes_box.contains(box.x1, box.y1)
        if inside and not any(padded.overlaps(other) for other in occupied):
            occupied.append(padded)
            return
        note.remove()
    dx, dy, side = options[0]
    note = ax.annotate(text, point, xytext=(dx, dy), textcoords="offset points", ha=side, fontsize=7, fontweight=weight, arrowprops=arrow)
    note.get_window_extent(renderer)
    occupied.append(Text.get_window_extent(note, renderer))


for k in np.argsort(-sent)[: min(5, len(sent))]:
    place_label(str(Z_SOURCES.index[k]), (source_scores[k, 0], source_scores[k, 1]), radius=float(np.sqrt(sizes[k] / np.pi)))
for k, neighbour in ((0, 1), (len(Z_TARGET) - 1, len(Z_TARGET) - 2)):
    away = 1 if target_scores[k, 0] >= target_scores[neighbour, 0] else -1
    place_label(str(Z_TARGET.index[k]), (target_scores[k, 0], target_scores[k, 1]), radius=5.0, bold=True, prefer=away)
figure_path = ctx.save_figure(fig, "ot_feature_space.png")

In [ ]:
feat = support["features"].sort_values("weight", ascending=True)
centre_f = CENTRE.reindex(feat.index)
scale_f = SCALE.reindex(feat.index)
lo_z = (feat["source_min"] - centre_f) / scale_f
hi_z = (feat["source_max"] - centre_f) / scale_f
span_z = hi_z - lo_z
fig, ax = plt.subplots(figsize=(8.4, 0.6 * len(feat) + 2.6))
for k, (name, row) in enumerate(feat.iterrows()):
    ax.hlines(k, lo_z[name] - RANGE_TOLERANCE * span_z[name], hi_z[name] + RANGE_TOLERANCE * span_z[name], color="0.78", linewidth=10)
    ax.hlines(k, lo_z[name], hi_z[name], color="0.3", linewidth=3)
    ax.plot(
        row["z_target"], k, marker="D", markersize=8, linestyle="none", markeredgecolor="0.0",
        markerfacecolor="0.0" if row["outside"] else "white", zorder=4,
    )
ax.axvline(-CLIP, color="0.4", linestyle=":", linewidth=1.0)
ax.axvline(CLIP, color="0.4", linestyle=":", linewidth=1.0)
ax.set_xscale("symlog", linthresh=CLIP)
x_limits = ax.get_xlim()
tick_values = [-100, -10, -1, 0, 1, 10, 100]
ax.set_xticks(tick_values)
ax.set_xticklabels([f"{value:g}" for value in tick_values])
ax.minorticks_off()
ax.set_xlim(x_limits)
ax.set_yticks(np.arange(len(feat)))
ax.set_yticklabels([f"{name} (weight {row['weight']:.2f})" for name, row in feat.iterrows()], fontsize=8)
ax.set_xlabel("Standardised feature value, not clipped (robust standard deviations from the reference median; linear within the clip bound, logarithmic beyond)")
ax.set_ylabel("Feature (importance weight)")
ax.set_title("Target against the range of the sources that send mass")
ax.grid(axis="y", visible=False)
handles = [
    Line2D([], [], color="0.78", linewidth=10, label="Range of the sources with the margin"),
    Line2D([], [], color="0.3", linewidth=3, label="Range of the sources that send mass"),
    Line2D([], [], color="0.0", marker="D", markerfacecolor="white", linestyle="none", label="Target, inside the band"),
    Line2D([], [], color="0.0", marker="D", markerfacecolor="0.0", linestyle="none", label="Target, outside the band"),
    Line2D([], [], color="0.4", linestyle=":", label="Clip bound of the cost"),
]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=2, fontsize=8)
fig.tight_layout()
figure_path = ctx.save_figure(fig, "ot_target_support.png")

## Saved results

In [ ]:
target_features = RAW_CLOUDS["primary"].loc[TARGET_YEARS[-1], cases.FEATURE_COLUMNS].astype(float).to_dict()
saved_baseline = ctx.save_table(baseline, "thailand_target_baseline", index=True)
overlap_result = {
    "statistic": overlap["statistic"],
    "p_value": overlap["p_value"],
    "n_perm": overlap["n_perm"],
    "null_median": float(np.median(null_stat)),
    "null_p95": float(np.quantile(null_stat, 0.95)),
    "features_in_test": test_features,
    "features_from_source_episodes": LACKING,
    "weights_in_test": {k: float(v) for k, v in w_test.items()},
    "reference_pool_rows": int(len(pool_z)),
    "seed": SEED_PERM,
}
saved_overlap = ctx.save_json(overlap_result, "overlap_test")

transport_result = {
    "sample": "primary",
    "n_episodes": int(len(primary)),
    "n_economies": int(per_economy.size),
    "weights": "weight_importance",
    "features": FEATURES,
    "epsilon": EPS_MAIN,
    "rho_source": RHO_SOURCE,
    "rho_target": None,
    "target_marginal": "enforced",
    "transported_mass": float(plan.mass),
    "effective_sample_size": {"episodes": float(ess_episodes), "economies": float(ess_economies)},
    "converged": bool(plan.converged),
    "overlap_test": {"statistic": overlap["statistic"], "p_value": overlap["p_value"], "n_perm": overlap["n_perm"], "features_in_test": test_features},
    "support": support_record(support),
    "target": {
        "iso3": ctx.target_iso3,
        "name": ctx.target_name,
        "scenario": scenario["name"],
        "capex_thb_bn": float(PRIMARY_ROW["capex_thb_bn"]),
        "cost_year": COST_YEAR,
        "fx_thb_per_usd": FX_COST,
        "gdp_usd_bn": GDP_COST,
        "capex_usd_bn": float(PRIMARY_ROW["capex_usd_bn"]),
        "capex_pct_gdp": float(PRIMARY_ROW["capex_pct_gdp"]),
        "cost_year_alternative": COST_YEAR_ALTERNATIVE,
        "capex_pct_gdp_alternative": float(PRIMARY_ROW["capex_pct_gdp_alternative"]),
        "receipts_pct_gdp": TARGET_RECEIPTS_SHARE,
        "receipts_window_years": [int(y) for y in window_years],
        "receipts_basis": "panel series of international tourism receipts; years without a panel value use the ratio of the panel to the baseline",
        "years": [int(y) for y in TARGET_YEARS],
        "features_last_target_year": target_features,
        "scenarios": scenario_table.drop(columns=["scenario"]).to_dict(orient="index"),
    },
    "att_pp": {
        "estimate": float(effect_pp),
        "bootstrap_percentiles": quantile_dict(boot_pp["draws"]),
        "conformal_interval": interval_record(pred_pp["interval"]),
        "predictive_percentiles": quantile_dict(pred_pp["draws"]),
        "predictive_method": PREDICTIVE_METHOD,
        "sampling_predictive_percentiles": quantile_dict(boot_pp["predictive_draws"]),
        "between_sd": float(boot_pp["between_sd"]),
        "measurement_sd": float(boot_pp["measurement_sd"]),
        "by_target_year": {str(y): float(v) for y, v in by_year["att_pp"].items()},
    },
    "att_rel_pct": {
        "estimate": float(effect_rel),
        "bootstrap_percentiles": quantile_dict(boot_rel["draws"]),
        "conformal_interval": interval_record(pred_rel["interval"]),
        "predictive_percentiles": quantile_dict(pred_rel["draws"]),
        "predictive_method": PREDICTIVE_METHOD,
        "sampling_predictive_percentiles": quantile_dict(boot_rel["predictive_draws"]),
        "between_sd": float(boot_rel["between_sd"]),
        "measurement_sd": float(boot_rel["measurement_sd"]),
        "by_target_year": {str(y): float(v) for y, v in by_year["att_rel_pct"].items()},
    },
    "predictive": {
        "method": PREDICTIVE_METHOD,
        "basis": "leave-one-economy-out prediction errors, resampled by economy, subtracted from the transported effect and rescaled about it to the conformal interval",
        "level": CONFORMAL_LEVEL,
        "calibrated": {"att_pp": bool(pred_pp["calibrated"]), "att_rel_pct": bool(pred_rel["calibrated"])},
        "scale": {"att_pp": float(pred_pp["scale"]), "att_rel_pct": float(pred_rel["scale"])},
        "n_draws": int(pred_pp["n_draws"]),
        "n_economies": int(pred_pp["n_groups"]),
        "n_episodes": int(pred_pp["n_cases"]),
    },
    "scenarios": SCENARIO_RESULTS,
    "validation": {
        "weights": "importance relearned in every fold from the training episodes",
        "n_folds": N_FOLDS,
        "n_folds_uniform_weights": N_UNIFORM_FOLDS,
        "importance_repeats": IMPORTANCE_REPEATS,
        "importance_configuration": "light",
        "n_nonconverged": int(loco.n_nonconverged + loco_rel.n_nonconverged),
        "n_log_domain": int(loco.n_log_domain),
        "n_log_domain_rel": int(loco_rel.n_log_domain),
        "n_eps_fallback": int(loco.n_eps_fallback),
        "n_eps_fallback_rel": int(loco_rel.n_eps_fallback),
        "ratio_note": loco.ratio_note,
    },
    "null": {
        "placebos": "comparable placebos of the sources",
        "n_placebos": n_placebos_all,
        "n_placebos_comparable": n_placebos_comparable,
    },
    "clip": CLIP,
    "n_clipped_sources": n_clipped_sources,
    "null_percentiles": {"att_pp": {k: float(v) for k, v in null_quantiles["att_pp"].items()}, "att_rel_pct": {k: float(v) for k, v in null_quantiles["att_rel_pct"].items()}},
    "n_boot": int(N_BOOT_TRANSPORT),
    "n_nonconverged_bootstrap": int(boot_pp["n_nonconverged"] + boot_rel["n_nonconverged"]),
    "n_log_domain_bootstrap": int(boot_pp["n_log_domain"] + boot_rel["n_log_domain"]),
    "plan_log_domain": PLAN_LOG_DOMAIN,
    "input_digests": INPUT_DIGESTS,
    "seeds": {"bootstrap": SEED_BOOT, "null draws": SEED_NULL, "overlap test": SEED_PERM, "validation": SEED_LOCO, "importance": SEED_IMPORTANCE},
}
saved_transport = ctx.save_json(transport_result, "thailand_transport")

## Checks

In [ ]:
assert plan.converged and np.isfinite(plan.pi).all() and (plan.pi >= 0).all()
assert plan.pi.shape == (len(primary), len(TARGET_YEARS))
assert plan.marginal_error < 1e-6, "marginal error relative to the total mass"
assert np.abs(plan.pi.sum(axis=0) - B_TARGET).max() <= 1e-6 * plan.mass, "the target marginal is enforced"
assert abs(plan.mass - B_TARGET.sum()) <= 1e-6 * B_TARGET.sum()
assert float(np.abs(Z_SOURCES.to_numpy()).max()) <= CLIP + 1e-12 and float(np.abs(Z_TARGET.to_numpy()).max()) <= CLIP + 1e-12
assert 1.0 <= ess_episodes <= len(primary) + 1e-9 and 1.0 <= ess_economies <= usage_by_economy.size + 1e-9
assert np.isfinite([effect_pp, effect_rel]).all()
assert set(loco.summary.index) == {"ot_weighted", "ot_uniform", "equal", "nn1", "nn3", "kernel"}
assert np.isfinite(loco.summary["rmse"]).all() and (loco.summary["rmse"] > 0).all()
assert list(loco.ratios.index) == ["ot_weighted / equal", "ot_weighted / ot_uniform"]
assert (loco.ratios["lo95"] <= loco.ratios["hi95"]).all() and (loco.ratios["lo80"] <= loco.ratios["hi80"]).all()
assert loco.n_nonconverged == 0 and loco_rel.n_nonconverged == 0
assert 0 <= loco.n_eps_fallback and 0 <= loco_rel.n_eps_fallback
assert set(loco_rel.summary.index) == set(loco.summary.index) and np.isfinite(loco_rel.summary["rmse"]).all() and (loco_rel.summary["rmse"] > 0).all()
assert np.allclose(fold_table.groupby("held_out_group")["weight"].sum(), 1.0) and (fold_table["weight"] >= 0).all()
assert set(fold_table["held_out_group"]) == {str(g) for g in GROUPS} and len(fold_weights.records) == len(set(GROUPS))
assert 0.0 < overlap["p_value"] <= 1.0

# The target description and the support check
assert np.isclose(float(PRIMARY_ROW["capex_pct_gdp"]), 100.0 * float(PRIMARY_ROW["capex_thb_bn"]) / (FX_COST * GDP_COST))
for name, spec in CLOUD_SPECS.items():
    assert np.allclose(RAW_CLOUDS[name]["capex_pct_gdp"].to_numpy(dtype=float), spec["share"]), f"the cost share of the cloud {name} is not constant"
assert np.isclose(float(window_share.mean()), TARGET_RECEIPTS_SHARE) and TARGET_RECEIPTS_SHARE > 0.0
assert (baseline.loc[baseline["receipts_basis"] == "panel", "basis_ratio"].isna()).all()
assert support["supported"] == (support["ess_ok"] and support["range_ok"])
assert abs(float(support["features"]["weight"].sum()) - 1.0) < 1e-9
assert support["reasons"] == [] if support["supported"] else len(support["reasons"]) >= 1
assert abs(support["ess"] - ess_episodes) < 1e-9
assert set(SCENARIO_RESULTS) == set(scenario_table.index) and SCENARIO_RESULTS[scenario["name"]]["primary"]

# The predictive draws and the intervals
assert np.isfinite(predictive.drop(columns="draw").to_numpy()).all() and len(predictive) == N_BOOT_TRANSPORT
assert (np.diff(pred_pp["percentiles"].to_numpy()) >= 0).all() and (np.diff(pred_rel["percentiles"].to_numpy()) >= 0).all()
for pred, estimate in ((pred_pp, effect_pp), (pred_rel, effect_rel)):
    interval = pred["interval"]
    assert interval["n"] == len(primary)
    if interval["status"] == "ok":
        assert interval["lower"] <= estimate <= interval["upper"]
        assert interval["guaranteed_level"] >= interval["level"] - 1e-12
        if pred["calibrated"]:
            half = float(np.quantile(np.abs(pred["draws"] - estimate), interval["level"]))
            assert abs(half - interval["half_width"]) <= 1e-6 * max(1.0, interval["half_width"]), "the draws do not reproduce the conformal interval"
    else:
        assert not pred["calibrated"] and np.isnan(interval["lower"])
assert np.isfinite(null_table.to_numpy()).all() and len(null_table) == N_NULL
assert n_placebos_comparable <= n_placebos_all and n_placebos_comparable >= len(Z_SOURCES)
assert list(boot_pp["percentiles"].index) == [5, 25, 50, 75, 95]
assert (np.diff(boot_pp["predictive_percentiles"].to_numpy()) >= 0).all()
assert abs(probs_pp.sum() - 1.0) < 1e-9 and abs(probs_rel.sum() - 1.0) < 1e-9

# The saved files
for name in (
    "loco_table", "loco_summary", "loco_ratios", "loco_table_rel", "loco_summary_rel", "loco_ratios_rel", "loco_fold_weights",
    "transport_plan", "thailand_transport_draws", "thailand_scenario_draws", "thailand_null_draws", "transport_usage",
    "transport_sensitivity", "thailand_target_baseline",
):
    assert ctx.table_path(name).is_file(), name
for name in ("thailand_transport.json", "overlap_test.json"):
    assert (ctx.results_dir / name).is_file(), name
saved = pd.read_csv(ctx.table_path("loco_summary"), index_col="method")
assert "rmse" in saved.columns and saved.index.name == "method"
assert list(pd.read_csv(ctx.table_path("thailand_transport_draws")).columns) == ["draw", "predictive_pp", "predictive_rel_pct", "transport_pp", "transport_rel_pct"]
assert list(pd.read_csv(ctx.table_path("thailand_scenario_draws")).columns) == ["scenario", "draw", "predictive_pp", "predictive_rel_pct"]
assert list(pd.read_csv(ctx.table_path("loco_fold_weights")).columns) == ["held_out_group", "feature", "weight", "fitted"]
assert list(pd.read_csv(ctx.table_path("thailand_null_draws")).columns) == ["null_pp", "null_rel_pct"]
saved_record = ctx.read_json("thailand_transport")
assert saved_record["support"]["supported"] == support["supported"] and set(saved_record["support"]["features"]) == set(FEATURES)
assert abs(saved_record["target"]["capex_pct_gdp"] - float(PRIMARY_ROW["capex_pct_gdp"])) < 1e-12 and set(saved_record["input_digests"]) == set(INPUT_TABLES)
print("Checks passed.")
print(f"Elapsed time of the notebook: {time.perf_counter() - NOTEBOOK_START:.0f} seconds")

## Scope and limits

The transported effect is a weighted average of the estimated effects of past openings, with weights that favour episodes resembling the target on the features that the importance analysis found to modify the effect. The effects of the cases are moved to the target economy; they are not rescaled for the size of a park, and the transported effect is not the total economic impact of a park. It is conditional on a park of the stated cost opening, and on the feature set, the importance weights and the entropic regularisation, whose influence the sensitivity table shows. It is a mean annual effect on international tourism receipts over the first five years after an opening, measured in percentage points of GDP and in percent of the counterfactual receipts.

A barycentric transport cannot exceed the largest effect among its sources. A target that is costlier, larger or more concentrated than every case is therefore an extrapolation that the method cannot make, and the support check reports it. The intervals of this notebook describe the target only inside the support that the check reports. The conformal interval is built from errors of episodes predicted at their own features, so it does not widen for a target that lies farther from the cases than the cases lie from each other. The overlap test fills the features that the reference table cannot describe from the source episodes, so it judges the economic context of the target against the sources and the cost share and the resort flag only through the distribution of the cases.

The source effects come from country-level synthetic controls with few pre-opening years, and the pandemic years are excluded from every estimation window. The yearly feature vectors of the target cloud come from the years before the pandemic. The proposal is not a commitment. The cost in US dollars is derived from the cost in baht with the exchange rate of one year, so it is an estimate, and the cost share holds that year's GDP; the alternative cost year shows how much the reading depends on the year. The baseline of the target is put on the basis of the panel series; the years without a panel value use the mean ratio of the panel to the baseline over the ratio years printed in the settings, so their receipts rest on that ratio.

Cases in one economy are not independent. The validation leaves out whole economies and relearns the feature weights in every fold from the remaining economies, so the weights of a fold carry no information about the effects of the left-out economy and the validation does not favour the weighted predictors through the learning of the weights. The folds use the light configuration of the importance estimator, which is cheaper than the configuration that gave the weights of the transport plan for the target economy, so the validation does not judge those weights exactly. A fold without enough training economies uses uniform weights, and the number of such folds is printed. The bootstrap resamples whole economies, and the effective sample size is reported for episodes and for economies. The intervals of the RMSE ratios describe the episodes at hand and are not a test; with few economies they are wide, and a ratio of RMSE values close to one does not separate the predictors. The conformal guarantee is exact only for exchangeable errors; errors of episodes of one economy and errors from overlapping training sets are not exactly exchangeable, so the stated coverage is approximate.